<a href="https://colab.research.google.com/github/kasianeno/User-Experimentation-and-A-B-Test-Dashboard/blob/main/AB_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[SQL code](https://docs.google.com/document/d/1bOJ_EfGep_lK14kWHjEwCT6nPVm7vTiJj5hd83P6k_s/edit?usp=sharing)

[Calculated table](https://drive.google.com/file/d/1Y0tLL0_hMTnbpEycIBGaX8W_ez80NTQN/view?usp=sharing)

[Tableau Dashboard](https://public.tableau.com/views/ABTest_17675349607140/ABTest?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

Python calculations down below.

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Pandas_tasks/main

df = pd.read_csv('df.csv')
df.head(100)

Mounted at /content/drive
/content/drive/MyDrive/Pandas_tasks/main


,date,ga_session_id,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,2773474671,Philippines,desktop,Asia,Paid Search,2,1,new account,1
1,2020-11-01,1609345343,Pakistan,desktop,Asia,Organic Search,2,1,new account,1
2,2020-11-01,2552014907,El Salvador,desktop,Americas,Social Search,2,1,new account,1
3,2020-11-02,4995450755,Romania,desktop,Europe,Organic Search,2,1,new account,1
4,2020-11-02,976397889,Austria,desktop,Europe,Social Search,2,1,new account,1
...,...,...,...,...,...,...,...,...,...,...
95,2020-11-28,310357037,Bangladesh,mobile,Asia,Direct,2,1,new account,1
96,2020-11-28,3662502697,Belgium,mobile,Europe,Paid Search,2,1,new account,1
97,2020-11-29,5788074854,Greece,desktop,Europe,Organic Search,2,2,new account,1
98,2020-11-19,5129795453,Denmark,desktop,Europe,Organic Search,2,1,new account,1


In [ ]:
#adding key metrics
metrics = [
    'add_payment_info',
    'add_shipping_info',
    'begin_checkout',
    'new account'
]

results = []

#grouping tests by unique sessions using loop
for test_name, test_data in df.groupby('test'):
    sessions = test_data.groupby('test_group')['ga_session_id'].nunique()

    #secondary loop for main calculations required the task
    for metric in metrics:

        #group sessions by required metrics and test group
        event_sessions = test_data[test_data['event_name'] == metric].groupby('test_group')['value'].sum()

        #adding numerator and denominator for upcoming calculations
        numerator_A = event_sessions.get(1, 0)
        numerator_B = event_sessions.get(2, 0)

        denominator_A = sessions.get(1, 0)
        denominator_B = sessions.get(2, 0)

        if denominator_A == 0 or denominator_B == 0:
            continue

        #total count
        events_count = np.array([numerator_A, numerator_B])
        sessions_count = np.array([denominator_A, denominator_B])

        #convertion rate for both groups
        cr_a = numerator_A / denominator_A
        cr_b = numerator_B / denominator_B

        if numerator_A == 0 and numerator_B == 0:
            p_value == np.nan and stat == np.nan
        else:
            stat, p_value = proportions_ztest(events_count, sessions_count)

        conv_a = numerator_A / denominator_A
        conv_b = numerator_B / denominator_B

        uplift = (conv_b - conv_a) / conv_a

        results.append({
            'test': test_name,
            'metric': metric,
            'sessions_a': denominator_A,
            'sessions_b': denominator_B,
            'events_a': numerator_A,
            'events_b': numerator_B,
            'conv_a': conv_a,
            'conv_b': conv_b,
            'uplift': uplift,
            'p_value': p_value if not np.isnan(p_value) else None,
            'z_stat': stat,
            'significant': p_value < 0.05
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["test", "metric"])

results_df.round(5)

,test,metric,sessions_a,sessions_b,events_a,events_b,conv_a,conv_b,uplift,p_value,z_stat,significant
0,1,add_payment_info,45362,45193,1988,2229,0.04383,0.04932,0.12542,0.00009,-3.92488,True
1,1,add_shipping_info,45362,45193,3034,3221,0.06688,0.07127,0.06560,0.00923,-2.60357,True
2,1,begin_checkout,45362,45193,3784,4021,0.08342,0.08897,0.06661,0.00289,-2.97878,True
3,1,new account,45362,45193,3823,3681,0.08428,0.08145,-0.03354,0.12286,1.54288,False
4,2,add_payment_info,50637,50244,2344,2409,0.04629,0.04795,0.03577,0.21461,-1.24099,False
5,2,add_shipping_info,50637,50244,3480,3510,0.06872,0.06986,0.01651,0.47798,-0.70956,False
6,2,begin_checkout,50637,50244,4262,4313,0.08417,0.08584,0.01988,0.34064,-0.95290,False
7,2,new account,50637,50244,4165,4184,0.08225,0.08327,0.01242,0.55600,-0.58879,False
8,3,add_payment_info,70047,70439,3623,3697,0.05172,0.05249,0.01475,0.52011,-0.64317,False
9,3,add_shipping_info,70047,70439,5298,5188,0.07563,0.07365,-0.02621,0.15744,1.41373,False


In [ ]:
#Same code with next metrics
metrics = df['device'].unique()

results = []
for test_name, test_data in df.groupby('test'):
    sessions = test_data.groupby('test_group')['ga_session_id'].nunique()

    for metric in metrics:
        event_sessions = test_data[test_data['device'] == metric].groupby('test_group')['value'].sum()

        numerator_A = event_sessions.get(1, 0)
        numerator_B = event_sessions.get(2, 0)

        denominator_A = sessions.get(1, 0)
        denominator_B = sessions.get(2, 0)

        if denominator_A == 0 or denominator_B == 0:
            continue

        events_count = np.array([numerator_A, numerator_B])
        sessions_count = np.array([denominator_A, denominator_B])

        cr_a = numerator_A / denominator_A
        cr_b = numerator_B / denominator_B

        if numerator_A == 0 and numerator_B == 0:
            p_value == np.nan and stat == np.nan
        else:
            stat, p_value = proportions_ztest(events_count, sessions_count)

        conv_a = numerator_A / denominator_A
        conv_b = numerator_B / denominator_B

        uplift = (conv_b - conv_a) / conv_a

        results.append({
            'test': test_name,
            'metric': metric,
            'sessions_a': denominator_A,
            'sessions_b': denominator_B,
            'events_a': numerator_A,
            'events_b': numerator_B,
            'conv_a': conv_a,
            'conv_b': conv_b,
            'uplift': uplift,
            'p_value': p_value if not np.isnan(p_value) else None,
            'z_stat': stat,
            'significant': p_value < 0.05
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["test", "metric"])

results_df.round(5)

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)


,test,metric,sessions_a,sessions_b,events_a,events_b,conv_a,conv_b,uplift,p_value,z_stat,significant
0,1,desktop,45362,45193,387602,407620,8.54464,9.01954,0.05558,NaN,NaN,False
1,1,mobile,45362,45193,270540,270195,5.96402,5.97869,0.00246,NaN,NaN,False
2,1,tablet,45362,45193,16248,13656,0.35819,0.30217,-0.15638,0.00000,17.92071,True
3,2,desktop,50637,50244,445367,440230,8.79529,8.76184,-0.00380,NaN,NaN,False
4,2,mobile,50637,50244,307230,290999,6.06730,5.79172,-0.04542,NaN,NaN,False
5,2,tablet,50637,50244,17297,17467,0.34159,0.34764,0.01773,0.04302,-2.02347,True
6,3,desktop,70047,70439,613503,619182,8.75845,8.79033,0.00364,NaN,NaN,False
7,3,mobile,70047,70439,403127,391138,5.75509,5.55286,-0.03514,NaN,NaN,False
8,3,tablet,70047,70439,22641,23332,0.32323,0.33124,0.02478,0.00138,-3.19974,True
9,4,desktop,105079,105141,782696,776525,7.44864,7.38556,-0.00847,NaN,NaN,False


In [ ]:
metrics = df['country'].unique()

results = []
for test_name, test_data in df.groupby('test'):
    sessions = test_data.groupby('test_group')['ga_session_id'].nunique()

    for metric in metrics:
        event_sessions = test_data[test_data['country'] == metric].groupby('test_group')['value'].sum()

        numerator_A = event_sessions.get(1, 0)
        numerator_B = event_sessions.get(2, 0)

        denominator_A = sessions.get(1, 0)
        denominator_B = sessions.get(2, 0)

        if denominator_A == 0 or denominator_B == 0:
            continue

        events_count = np.array([numerator_A, numerator_B])
        sessions_count = np.array([denominator_A, denominator_B])

        cr_a = numerator_A / denominator_A
        cr_b = numerator_B / denominator_B

        if numerator_A == 0 and numerator_B == 0:
            p_value == np.nan and stat == np.nan
        else:
            stat, p_value = proportions_ztest(events_count, sessions_count)

        conv_a = numerator_A / denominator_A
        conv_b = numerator_B / denominator_B

        uplift = (conv_b - conv_a) / conv_a

        results.append({
            'test': test_name,
            'metric': metric,
            'sessions_a': denominator_A,
            'sessions_b': denominator_B,
            'events_a': numerator_A,
            'events_b': numerator_B,
            'conv_a': conv_a,
            'conv_b': conv_b,
            'uplift': uplift,
            'p_value': p_value if not np.isnan(p_value) else None,
            'z_stat': stat,
            'significant': p_value < 0.05
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["test", "metric"])

results_df.round(5)

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/proportion.py:1024: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)


,test,metric,sessions_a,sessions_b,events_a,events_b,conv_a,conv_b,uplift,p_value,z_stat,significant
81,1,(not set),45362,45193,5117,5482,0.11280,0.12130,0.07534,0.00007,-3.97752,True
46,1,Albania,45362,45193,132,257,0.00291,0.00569,0.95425,0.00000,-6.38831,True
61,1,Algeria,45362,45193,399,318,0.00880,0.00704,-0.20003,0.00282,2.98688,True
6,1,Argentina,45362,45193,1795,1826,0.03957,0.04040,0.02107,0.52191,-0.64040,False
73,1,Armenia,45362,45193,358,154,0.00789,0.00341,-0.56822,0.00000,8.99887,True
...,...,...,...,...,...,...,...,...,...,...,...,...
430,4,United Kingdom,105079,105141,42000,42078,0.39970,0.40021,0.00127,0.81276,-0.23687,False
431,4,United States,105079,105141,584322,582487,5.56079,5.54006,-0.00373,NaN,NaN,False
380,4,Uruguay,105079,105141,722,926,0.00687,0.00881,0.28179,0.00000,-5.03297,True
381,4,Venezuela,105079,105141,416,416,0.00396,0.00396,-0.00059,0.99320,0.00852,False


## Summary

Conducted A/B testing (4 experiments) to optimize the checkout flow.

Identified **statistically significant uplift** of **up to +12.5%** in add_payment_info and **+6–7% across key funnel stages** (Test 1).

Detected **negative impact** in Test 3–4 → conversion **decreased by up to -3.3%** at the checkout stage.

Performed **device-level segmentation** → found **-15.6% drop** in conversion for tablet users.

Discovered variability across countries → identified markets with up **to +95% uplift** (low-base effect)

Validated statistical test assumptions → identified edge cases (NaN p-values) caused by zero-event scenarios

Provided data-driven recommendations → approved rollout for Test 1 and rejected other variants